# The Alchemist's Codex — Solution Walkthrough

**Original solution by Cowile — annotated for instructors**

---

## Problem Recap

We have an alchemy system where combining two items produces a result:

```
item1 + item2 → result
```

- **150 known recipes** (training data)
- **70 unknown recipes** (test data) — we know the inputs but not the outputs
- **70 candidate results** — the possible outputs, each used **exactly once**

The mapping is **bijective**: every test pair maps to one unique candidate, and every candidate is used exactly once. This makes it a **bipartite matching** problem on top of a prediction problem.

---

## Solution Strategy (3 Key Ideas)

This solution improves on the baseline in three ways:

| Step | Baseline | This Solution |
|------|----------|---------------|
| **Scoring** | Cosine similarity of frozen BERT embeddings | Fine-tuned **cross-encoder** that reads the full sentence |
| **Training** | No training at all | Contrastive training with **in-batch negatives** |
| **Matching** | Greedy (pick best unused) | **Hungarian algorithm** (globally optimal 1-to-1 assignment) |

Let's walk through each part.

In [ ]:
!pip install -q transformers kagglehub

---

## Part 1: Setup & Imports

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from scipy.optimize import linear_sum_assignment

# Fix all random seeds for reproducibility
random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---

## Part 2: Load & Explore the Data

In [ ]:
import kagglehub

DATA = kagglehub.dataset_download("sattamjaltwaim/ioai-alchemy")
print(f"Dataset path: {DATA}")

train_df = pd.read_csv(f'{DATA}/train.csv')
test_df  = pd.read_csv(f'{DATA}/test.csv')
candidates = sorted(pd.read_csv(f'{DATA}/candidates.csv')['result'].unique().tolist())

print(f"Training recipes : {len(train_df)}")
print(f"Test pairs       : {len(test_df)}")
print(f"Candidate results: {len(candidates)}")
print()
print("Sample training recipes:")
train_df.head()

---

## Part 3: Data Augmentation (Exploit Commutativity)

A crucial property from the data description:

> **Commutativity**: `item1 + item2` and `item2 + item1` produce the same result.

So `fire + earth = lava` also means `earth + fire = lava`.

We **double our training data** by adding the swapped version of every recipe (where `item1 ≠ item2`). This teaches the model that order doesn't matter.

In [ ]:
augmented = []
for _, row in train_df.iterrows():
    # Original: item1 + item2 → result
    augmented.append({'item1': row['item1'], 'item2': row['item2'], 'result': row['result']})
    # Swapped:  item2 + item1 → result  (skip if both items are the same)
    if row['item1'] != row['item2']:
        augmented.append({'item1': row['item2'], 'item2': row['item1'], 'result': row['result']})

train_aug = pd.DataFrame(augmented)
print(f"Original training size: {len(train_df)}")
print(f"After augmentation:     {len(train_aug)}")

We also collect every result we've ever seen (from train + candidates) into one pool. This will be used later to sample hard distractors during training.

In [ ]:
all_known_results = sorted(list(set(train_df['result'].unique()) | set(candidates)))
print(f"Total unique results across train + candidates: {len(all_known_results)}")

---

## Part 4: The Sentence Template

Instead of embedding items and results separately (like the baseline does), we form a **complete English sentence** and let BERT read the whole thing:

```
"combining fire and earth creates lava"
```

This is the **cross-encoder** idea: BERT sees all three elements (item1, item2, result) together in one sentence, so it can reason about whether the combination makes sense. This is much more powerful than comparing separate embeddings.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
template = 'combining {a} and {b} creates {r}'

# Example
print(template.format(a='fire', b='earth', r='lava'))

---

## Part 5: The Cross-Encoder Model

### What is a cross-encoder?

A **cross-encoder** takes a full sentence as input and outputs a single score: "how plausible is this sentence?"

```
"combining fire and earth creates lava"   → score: 0.92  ✓ (high = plausible)
"combining fire and earth creates cheese"  → score: 0.03  ✗ (low = implausible)
```

Compare this to the **baseline's bi-encoder** approach, which embeds `"fire earth"` and `"lava"` separately and measures cosine similarity. The cross-encoder is stronger because BERT can attend across all tokens simultaneously.

### Architecture

```
Input sentence  →  BERT  →  [CLS] token embedding (768-d)  →  MLP head  →  scalar score
```

### Freezing strategy

We freeze the embedding layer and the first 7 (of 12) transformer layers. Only the top 5 layers and the MLP head are trained. This prevents overfitting on our small dataset (only ~300 training examples) while still allowing the model to adapt.

In [ ]:
class CrossEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        # Freeze embeddings + first 7 transformer layers to prevent overfitting
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for layer in self.bert.encoder.layer[:7]:
            for p in layer.parameters():
                p.requires_grad = False

        # Small MLP head: [CLS] embedding → plausibility score
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(768, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def score(self, texts):
        """Take a list of sentences, return a score for each one."""
        enc = tokenizer(
            texts, padding='max_length', truncation=True,
            max_length=48, return_tensors='pt'
        )
        ids  = enc['input_ids'].to(device)
        mask = enc['attention_mask'].to(device)
        cls_emb = self.bert(ids, mask).last_hidden_state[:, 0]  # [CLS] token
        return self.head(cls_emb).squeeze(-1)


model = CrossEncoder().to(device)

# Quick sanity check: count trainable vs frozen parameters
total    = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,} ({100*trainable/total:.1f}%)")

---

## Part 6: Training — Contrastive Learning with In-Batch Negatives

### The core idea

We don't train a classifier with 70 fixed output classes. Instead, we train the model to **rank** correct results higher than incorrect ones.

For each training batch of size `n`:

1. Sample `n` recipes (e.g., 16)
2. Build a **candidate pool** of `n` results — the correct answers plus random distractors
3. Score **every pair × every candidate** → an `n × n` score matrix
4. Apply **cross-entropy loss**: each row should peak at the column of its correct result

### Visualizing the score matrix

```
                    candidate_0   candidate_1   candidate_2   candidate_3
  fire + earth        0.1           0.9*          0.2           0.0
  water + air          0.3           0.1           0.8*          0.0
  bird + fire          0.0           0.1           0.1           0.9*
  ice + sun            0.8*          0.0           0.1           0.0
```

Each `*` marks the correct result. Cross-entropy loss pushes the model to put high scores on the diagonal targets.

### Why this works

- The model learns to **distinguish** correct from incorrect results, not just memorize
- Other batch items act as **hard negatives** (they're all real alchemy results, just wrong for this pair)
- This is the same technique used in training retrieval models (e.g., DPR, ColBERT)

### Optimizer setup

We use **differential learning rates**: a small LR for the BERT backbone (which is already pretrained) and a larger LR for the MLP head (which is randomly initialized).

In [ ]:
# Differential learning rates: fine-tune BERT gently, train the head aggressively
backbone_params = [
    p for name, p in model.named_parameters()
    if p.requires_grad and 'head' not in name
]

optimizer = torch.optim.AdamW([
    {'params': backbone_params,          'lr': 2e-5},   # BERT top layers: small LR
    {'params': model.head.parameters(),  'lr': 1e-3},   # MLP head: 50x larger LR
])

# Linear warmup then linear decay — standard for BERT fine-tuning
total_steps = 5 * 60  # 5 epochs × 60 steps per epoch = 300 steps
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=30, num_training_steps=total_steps
)

### The training loop

In [ ]:
EPOCHS = 5
STEPS_PER_EPOCH = 60
BATCH_SIZE = 16

for epoch in range(EPOCHS):
    model.train()
    losses = []

    for _ in range(STEPS_PER_EPOCH):
        # --- Step A: Sample a batch of training recipes ---
        batch = train_aug.sample(BATCH_SIZE)
        items_a = batch['item1'].tolist()
        items_b = batch['item2'].tolist()
        correct_results = batch['result'].tolist()
        n = len(items_a)

        # --- Step B: Build the candidate pool for this batch ---
        # Start with all correct answers, then pad with random distractors
        candidate_pool = list(set(correct_results))
        while len(candidate_pool) < n:
            distractor = random.choice(all_known_results)
            if distractor not in candidate_pool:
                candidate_pool.append(distractor)
        candidate_pool = candidate_pool[:n]
        random.shuffle(candidate_pool)

        # Target: for each recipe, which index in candidate_pool is its correct result?
        target = torch.tensor(
            [candidate_pool.index(r) for r in correct_results],
            device=device
        )

        # --- Step C: Build the n×n score matrix ---
        # For every (pair_i, candidate_j) combination, form a sentence and score it
        texts = [
            template.format(a=items_a[i], b=items_b[i], r=candidate_pool[j])
            for i in range(n)
            for j in range(n)
        ]
        score_matrix = model.score(texts).view(n, n)

        # --- Step D: Cross-entropy loss over the score matrix ---
        # Each row is treated as a classification: pick the right candidate column
        loss = F.cross_entropy(score_matrix, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    print(f'Epoch {epoch + 1}/{EPOCHS}  —  avg loss: {np.mean(losses):.4f}')

---

## Part 7: Inference — Score Every (Test Pair, Candidate) Combination

Now we use the trained model to score all 70 × 70 = 4,900 possible (test pair, candidate) combinations.

Because the combination is commutative (`A + B = B + A`), we score **both orderings** and average them. This makes the predictions more robust:

```
score = average(
    score("combining A and B creates R"),
    score("combining B and A creates R")
)
```

In [ ]:
model.eval()
score_mat = np.zeros((len(test_df), len(candidates)))

for idx, row in test_df.iterrows():
    i = test_df.index.get_loc(idx)

    # Score both orderings of the pair against every candidate
    forward_texts = [template.format(a=row['item1'], b=row['item2'], r=c) for c in candidates]
    reverse_texts = [template.format(a=row['item2'], b=row['item1'], r=c) for c in candidates]

    with torch.no_grad():
        fwd_scores = model.score(forward_texts).cpu().numpy()
        rev_scores = model.score(reverse_texts).cpu().numpy()

    score_mat[i] = (fwd_scores + rev_scores) / 2

print(f"Score matrix shape: {score_mat.shape}  (70 test pairs × 70 candidates)")

---

## Part 8: Hungarian Matching — Globally Optimal Assignment

### Why not just pick the highest score per row?

If we greedily assign each test pair its top-scoring candidate (like the baseline does), **two pairs might fight over the same candidate**, and the loser gets pushed to a worse choice.

### The Hungarian algorithm

The **Hungarian algorithm** (`scipy.optimize.linear_sum_assignment`) finds the **globally optimal 1-to-1 assignment** that maximizes total score across all 70 pairs simultaneously.

Think of it like this:
- **Greedy**: each person picks their favorite seat one at a time — early pickers win, latecomers get leftovers
- **Hungarian**: a coordinator assigns everyone to seats so that **total satisfaction is maximized**

We negate the score matrix because `linear_sum_assignment` minimizes cost, and we want to maximize score.

In [ ]:
# Negate because linear_sum_assignment minimizes, but we want to maximize
_, col_assignments = linear_sum_assignment(-score_mat)

predictions = [candidates[c] for c in col_assignments]

print("Sample predictions:")
for i in range(5):
    row = test_df.iloc[i]
    print(f"  {row['item1']} + {row['item2']}  →  {predictions[i]}")

---

## Part 9: Generate Submission

In [ ]:
# ============================================
# DO NOT MODIFY -- Submission Generator
# ============================================
submission = pd.DataFrame({'Id': test_df['Id'], 'result': predictions})
submission.to_csv('submission.csv', index=False)
print(f"Submission shape: {submission.shape}")
submission.head(10)

---

## Summary: What Makes This Solution Better Than the Baseline?

| Aspect | Baseline (~60–70) | This Solution (~85+) |
|--------|-------------------|---------------------|
| **Embedding** | Bi-encoder: embed pair and result separately | Cross-encoder: BERT reads the full sentence together |
| **Training** | None — uses frozen BERT | Fine-tuned on training recipes with contrastive loss |
| **Data augmentation** | None | Swap `item1 ↔ item2` to exploit commutativity |
| **Matching** | Greedy (pick best unused) | Hungarian algorithm (globally optimal) |
| **Inference averaging** | Single direction | Average forward + reverse scores |

### Ideas to explore further

- **More epochs or larger batches** — the loss is still decreasing at epoch 5
- **Unfreeze more layers** — risky with small data, but worth experimenting
- **Ensembling** — train multiple models with different seeds and average their score matrices
- **Better templates** — try different phrasings like `"{a} plus {b} equals {r}"`
- **Train-set validation** — hold out some training recipes to tune hyperparameters